# Tiền xử lý dữ liệu cho hệ hỗ trợ quyết định dự đoán lương

Notebook này thực hiện tiền xử lý chính thức sau bước EDA cho bộ dữ liệu `250K Job Salary Prediction Dataset`.

Mục tiêu:

1. Đọc dữ liệu từ file `data.csv` nằm cùng thư mục với notebook.
2. Kiểm tra lại chất lượng dữ liệu trước khi mô hình hóa.
3. Chuẩn hóa tên cột và chuẩn hóa chuỗi phân loại.
4. Tạo biến phụ có kiểm soát, ví dụ `experience_group`.
5. Tách dữ liệu thành train/test trước khi fit bộ tiền xử lý để tránh rò rỉ dữ liệu.
6. Mã hóa biến phân loại bằng One-Hot Encoding.
7. Điền khuyết dữ liệu bằng `SimpleImputer`, dù dữ liệu hiện tại không có missing values.
8. Chuẩn hóa biến số bằng `StandardScaler` nếu dùng mô hình tuyến tính hoặc mô hình nhạy với thang đo.
9. Lưu pipeline tiền xử lý, ma trận dữ liệu đã xử lý và báo cáo tiền xử lý.

Lưu ý: dữ liệu gốc là dữ liệu synthetic, phù hợp để xây dựng nguyên mẫu DSS và minh họa quy trình học máy.

In [1]:
# ============================================================
# CELL 1. IMPORT THƯ VIỆN
# ============================================================

from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from scipy import sparse
import joblib

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [2]:
# ============================================================
# CELL 2. CẤU HÌNH CÓ THỂ CHỈNH
# ============================================================

DATA_PATH = Path("data.csv")
TARGET = "salary"

OUTPUT_DIR = Path("preprocessing_salary_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Chia dữ liệu
TEST_SIZE = 0.20
RANDOM_STATE = 42

# Biến mục tiêu
# TARGET_MODE = "raw": dự đoán trực tiếp salary
# TARGET_MODE = "log": dự đoán log1p(salary), khi đánh giá cần quy đổi ngược bằng expm1
TARGET_MODE = "raw"

# Xử lý ngoại lai của y_train. Không dùng thông tin từ y_test để fit cận ngoại lai.
# OUTLIER_MODE = "keep": giữ nguyên
# OUTLIER_MODE = "winsorize": chặn y_train theo cận IQR tính trên train
# OUTLIER_MODE = "remove": loại quan sát ngoại lai khỏi train, giữ nguyên test để đánh giá khách quan
OUTLIER_MODE = "keep"
IQR_K = 1.5

# Có tạo biến nhóm kinh nghiệm không?
CREATE_EXPERIENCE_GROUP = True

# Có đưa experience_group vào mô hình không?
# Nếu True: mô hình dùng cả experience_years và experience_group.
# Nếu muốn mô hình gọn hơn, đặt False.
USE_EXPERIENCE_GROUP_AS_FEATURE = False

# Chuẩn hóa biến số.
# True phù hợp với Linear Regression, Ridge, Lasso, KNN, Neural Network.
# Với Random Forest / Gradient Boosting, chuẩn hóa không bắt buộc nhưng vẫn không gây lỗi.
SCALE_NUMERIC = True

# Có lưu bản dữ liệu đã xử lý dạng CSV dense không?
# Với bộ 250k dòng, file CSV có thể lớn. Mặc định False, lưu sparse .npz hiệu quả hơn.
SAVE_DENSE_PROCESSED_CSV = False

# Các cột chắc chắn không đưa vào mô hình, nếu tồn tại.
EXCLUDE_FEATURES = []

In [3]:
# ============================================================
# CELL 3. HÀM TIỆN ÍCH
# ============================================================

def standardize_columns(data):
    """
    Chuẩn hóa tên cột:
    - bỏ khoảng trắng đầu/cuối
    - chuyển về chữ thường
    - thay khoảng trắng bằng dấu gạch dưới
    """
    data = data.copy()
    data.columns = (
        data.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
    )
    return data


def clean_categorical_strings(data):
    """
    Chuẩn hóa biến dạng chuỗi:
    - ép về string nếu là object/category
    - bỏ khoảng trắng đầu/cuối
    - thay chuỗi rỗng bằng NaN
    """
    data = data.copy()
    cat_cols = data.select_dtypes(include=["object", "category"]).columns.tolist()

    for col in cat_cols:
        data[col] = data[col].astype("string").str.strip()
        data[col] = data[col].replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})

    return data


def add_experience_group(data, experience_col="experience_years"):
    """
    Tạo nhóm kinh nghiệm phục vụ DSS và benchmark.
    Biến này có thể dùng hoặc không dùng trong mô hình dự đoán.
    """
    data = data.copy()

    if experience_col not in data.columns:
        return data

    bins = [-1, 2, 5, 10, 15, 20, np.inf]
    labels = [
        "0-2 years",
        "3-5 years",
        "6-10 years",
        "11-15 years",
        "16-20 years",
        "20+ years"
    ]

    data["experience_group"] = pd.cut(
        data[experience_col],
        bins=bins,
        labels=labels,
        include_lowest=True
    ).astype("string")

    return data


def split_feature_types(data, target, exclude_features=None):
    """
    Tự động tách biến số và biến phân loại.
    Hàm không phụ thuộc cứng vào bộ dữ liệu hiện tại.
    """
    exclude_features = exclude_features or []

    feature_cols = [
        c for c in data.columns
        if c != target and c not in exclude_features
    ]

    numeric_features = data[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
    categorical_features = [c for c in feature_cols if c not in numeric_features]

    return feature_cols, numeric_features, categorical_features


def iqr_bounds(y, k=1.5):
    """
    Tính cận ngoại lai theo quy tắc IQR trên một vector y.

    Q1 = percentile 25%
    Q3 = percentile 75%
    IQR = Q3 - Q1
    lower = Q1 - k * IQR
    upper = Q3 + k * IQR
    """
    y = pd.Series(y).dropna().astype(float)
    q1 = y.quantile(0.25)
    q3 = y.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return lower, upper


def transform_target(y, mode="raw"):
    """
    Biến đổi biến mục tiêu.
    - raw: giữ nguyên salary
    - log: dùng log1p(salary)
    """
    y = pd.Series(y).astype(float)

    if mode == "raw":
        return y

    if mode == "log":
        if (y < 0).any():
            raise ValueError("Không thể dùng log1p vì target có giá trị âm.")
        return np.log1p(y)

    raise ValueError("TARGET_MODE chỉ nhận 'raw' hoặc 'log'.")


def inverse_transform_target(y_transformed, mode="raw"):
    """
    Quy đổi ngược biến mục tiêu khi cần diễn giải dự báo.
    """
    y_transformed = np.asarray(y_transformed, dtype=float)

    if mode == "raw":
        return y_transformed

    if mode == "log":
        return np.expm1(y_transformed)

    raise ValueError("TARGET_MODE chỉ nhận 'raw' hoặc 'log'.")


def make_onehot_encoder():
    """
    Tạo OneHotEncoder tương thích với nhiều phiên bản scikit-learn.
    scikit-learn mới dùng sparse_output, phiên bản cũ dùng sparse.
    """
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)


def get_feature_names(preprocessor, numeric_features, categorical_features):
    """
    Lấy tên đặc trưng sau ColumnTransformer.
    """
    names = []

    if len(numeric_features) > 0:
        names.extend(numeric_features)

    if len(categorical_features) > 0:
        cat_pipeline = preprocessor.named_transformers_["categorical"]
        onehot = cat_pipeline.named_steps["onehot"]
        cat_names = onehot.get_feature_names_out(categorical_features).tolist()
        names.extend(cat_names)

    return names


def save_json(obj, path):
    path = Path(path)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def display_shape(name, obj):
    print(f"{name}: shape = {obj.shape}")

In [4]:
# ============================================================
# CELL 4. ĐỌC DỮ LIỆU
# ============================================================

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Không tìm thấy {DATA_PATH}. Hãy đặt file data.csv cùng thư mục với notebook."
    )

raw_df = pd.read_csv(DATA_PATH)
df = standardize_columns(raw_df)
df = clean_categorical_strings(df)

print("Kích thước dữ liệu gốc:", df.shape)
print("Tên cột:")
print(df.columns.tolist())

display(df.head())

Kích thước dữ liệu gốc: (250000, 10)
Tên cột:
['job_title', 'experience_years', 'education_level', 'skills_count', 'industry', 'company_size', 'location', 'remote_work', 'certifications', 'salary']


,job_title,experience_years,education_level,skills_count,industry,company_size,location,remote_work,certifications,salary
0,AI Engineer,10,Bachelor,2,Healthcare,Medium,India,Hybrid,2,109413
1,Data Analyst,5,Bachelor,17,Telecom,Small,Australia,No,0,93764
2,Frontend Developer,18,PhD,4,Media,Medium,Singapore,No,1,148123
3,Business Analyst,19,PhD,13,Retail,Medium,Canada,Yes,0,189123
4,Product Manager,15,Bachelor,7,Manufacturing,Large,Sweden,Yes,0,165069


In [5]:
# ============================================================
# CELL 5. KIỂM TRA CHẤT LƯỢNG DỮ LIỆU TRƯỚC TIỀN XỬ LÝ
# ============================================================

if TARGET not in df.columns:
    raise ValueError(f"Không tìm thấy biến mục tiêu: {TARGET}")

quality_report = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(df[c].dtype) for c in df.columns],
    "missing_count": [int(df[c].isna().sum()) for c in df.columns],
    "missing_rate": [float(df[c].isna().mean()) for c in df.columns],
    "n_unique": [int(df[c].nunique(dropna=True)) for c in df.columns],
})

n_duplicate = int(df.duplicated().sum())
quality_report["duplicate_rows_total"] = n_duplicate

quality_path = OUTPUT_DIR / "01_preprocessing_quality_report.csv"
quality_report.to_csv(quality_path, index=False, encoding="utf-8-sig")

print("Số dòng trùng lặp:", n_duplicate)
print("Đã lưu:", quality_path)
display(quality_report)

Số dòng trùng lặp: 0
Đã lưu: preprocessing_salary_outputs\01_preprocessing_quality_report.csv


,column,dtype,missing_count,missing_rate,n_unique,duplicate_rows_total
0,job_title,string,0,0.0000,12,0
1,experience_years,int64,0,0.0000,21,0
2,education_level,string,0,0.0000,5,0
3,skills_count,int64,0,0.0000,19,0
4,industry,string,0,0.0000,10,0
5,company_size,string,0,0.0000,5,0
6,location,string,0,0.0000,10,0
7,remote_work,string,0,0.0000,3,0
8,certifications,int64,0,0.0000,6,0
9,salary,int64,0,0.0000,118956,0


In [6]:
# ============================================================
# CELL 6. LÀM SẠCH CƠ BẢN
# ============================================================

# Loại dòng trùng nếu có. Với dữ liệu hiện tại, bước này không làm mất dòng nào.
df_clean = df.drop_duplicates().copy()

# Đảm bảo salary là số.
df_clean[TARGET] = pd.to_numeric(df_clean[TARGET], errors="coerce")

# Loại các dòng không có target vì mô hình supervised không học được nếu thiếu y.
before = len(df_clean)
df_clean = df_clean.dropna(subset=[TARGET]).copy()
after = len(df_clean)

print(f"Số dòng trước khi loại target missing: {before:,}")
print(f"Số dòng sau khi loại target missing: {after:,}")
print(f"Số dòng bị loại: {before - after:,}")

# Kiểm tra target dương nếu muốn dùng log.
if TARGET_MODE == "log" and (df_clean[TARGET] < 0).any():
    raise ValueError("TARGET_MODE='log' nhưng salary có giá trị âm.")

display(df_clean.head())

Số dòng trước khi loại target missing: 250,000
Số dòng sau khi loại target missing: 250,000
Số dòng bị loại: 0


,job_title,experience_years,education_level,skills_count,industry,company_size,location,remote_work,certifications,salary
0,AI Engineer,10,Bachelor,2,Healthcare,Medium,India,Hybrid,2,109413
1,Data Analyst,5,Bachelor,17,Telecom,Small,Australia,No,0,93764
2,Frontend Developer,18,PhD,4,Media,Medium,Singapore,No,1,148123
3,Business Analyst,19,PhD,13,Retail,Medium,Canada,Yes,0,189123
4,Product Manager,15,Bachelor,7,Manufacturing,Large,Sweden,Yes,0,165069


In [7]:
# ============================================================
# CELL 7. TẠO BIẾN PHỤ CÓ KIỂM SOÁT
# ============================================================

if CREATE_EXPERIENCE_GROUP:
    df_model = add_experience_group(df_clean, experience_col="experience_years")
else:
    df_model = df_clean.copy()

# Nếu chỉ dùng experience_group cho benchmark, không đưa vào mô hình.
exclude_features = list(EXCLUDE_FEATURES)

if CREATE_EXPERIENCE_GROUP and not USE_EXPERIENCE_GROUP_AS_FEATURE:
    exclude_features.append("experience_group")

feature_cols, numeric_features_all, categorical_features_all = split_feature_types(
    data=df_model,
    target=TARGET,
    exclude_features=exclude_features
)

print("Tổng số biến đầu vào dùng cho mô hình:", len(feature_cols))
print("Biến số:", numeric_features_all)
print("Biến phân loại:", categorical_features_all)
print("Biến bị loại khỏi mô hình:", exclude_features)

display(df_model[feature_cols + [TARGET]].head())

Tổng số biến đầu vào dùng cho mô hình: 9
Biến số: ['experience_years', 'skills_count', 'certifications']
Biến phân loại: ['job_title', 'education_level', 'industry', 'company_size', 'location', 'remote_work']
Biến bị loại khỏi mô hình: ['experience_group']


,job_title,experience_years,education_level,skills_count,industry,company_size,location,remote_work,certifications,salary
0,AI Engineer,10,Bachelor,2,Healthcare,Medium,India,Hybrid,2,109413
1,Data Analyst,5,Bachelor,17,Telecom,Small,Australia,No,0,93764
2,Frontend Developer,18,PhD,4,Media,Medium,Singapore,No,1,148123
3,Business Analyst,19,PhD,13,Retail,Medium,Canada,Yes,0,189123
4,Product Manager,15,Bachelor,7,Manufacturing,Large,Sweden,Yes,0,165069


In [8]:
# ============================================================
# CELL 8. TÁCH X, y VÀ CHIA TRAIN/TEST
# ============================================================

X = df_model[feature_cols].copy()
y_raw = df_model[TARGET].copy()

# Stratify nhẹ theo job_title và location nếu có, giúp train/test giữ cấu trúc nhóm chính.
# Nếu một nhóm quá ít quan sát, tự động không stratify.
stratify_key = None

possible_stratify_cols = [c for c in ["job_title", "location"] if c in X.columns]

if len(possible_stratify_cols) > 0:
    candidate_key = X[possible_stratify_cols].astype(str).agg("__".join, axis=1)
    group_counts = candidate_key.value_counts()

    if group_counts.min() >= 2:
        stratify_key = candidate_key
        print("Dùng stratify theo:", possible_stratify_cols)
    else:
        print("Không dùng stratify vì có nhóm quá nhỏ.")
else:
    print("Không có cột phù hợp để stratify.")

X_train, X_test, y_train_raw, y_test_raw = train_test_split(
    X,
    y_raw,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=stratify_key
)

print("Kích thước sau khi chia dữ liệu:")
display_shape("X_train", X_train)
display_shape("X_test", X_test)
print("y_train:", y_train_raw.shape)
print("y_test:", y_test_raw.shape)

Dùng stratify theo: ['job_title', 'location']
Kích thước sau khi chia dữ liệu:
X_train: shape = (200000, 9)
X_test: shape = (50000, 9)
y_train: (200000,)
y_test: (50000,)


In [9]:
# ============================================================
# CELL 9. XỬ LÝ NGOẠI LAI CỦA TARGET TRÊN TRAIN SET
# ============================================================

# Cận ngoại lai chỉ fit trên y_train để tránh dùng thông tin từ test.
lower_y, upper_y = iqr_bounds(y_train_raw, k=IQR_K)

outlier_train_flag = (y_train_raw < lower_y) | (y_train_raw > upper_y)
outlier_test_flag = (y_test_raw < lower_y) | (y_test_raw > upper_y)

outlier_report = pd.DataFrame({
    "set": ["train", "test"],
    "lower_bound_fit_on_train": [lower_y, lower_y],
    "upper_bound_fit_on_train": [upper_y, upper_y],
    "n": [len(y_train_raw), len(y_test_raw)],
    "n_outliers": [int(outlier_train_flag.sum()), int(outlier_test_flag.sum())],
    "outlier_rate": [float(outlier_train_flag.mean()), float(outlier_test_flag.mean())]
})

print("OUTLIER_MODE:", OUTLIER_MODE)
display(outlier_report)
outlier_report.to_csv(OUTPUT_DIR / "02_target_outlier_report_fit_on_train.csv", index=False, encoding="utf-8-sig")

# Áp dụng chế độ xử lý ngoại lai cho train.
# Test giữ nguyên để đánh giá khách quan khả năng dự đoán trên dữ liệu mới.
if OUTLIER_MODE == "keep":
    X_train_final = X_train.copy()
    y_train_final_raw = y_train_raw.copy()

elif OUTLIER_MODE == "winsorize":
    X_train_final = X_train.copy()
    y_train_final_raw = y_train_raw.clip(lower=lower_y, upper=upper_y)

elif OUTLIER_MODE == "remove":
    keep_mask = ~outlier_train_flag
    X_train_final = X_train.loc[keep_mask].copy()
    y_train_final_raw = y_train_raw.loc[keep_mask].copy()

else:
    raise ValueError("OUTLIER_MODE chỉ nhận 'keep', 'winsorize' hoặc 'remove'.")

X_test_final = X_test.copy()
y_test_final_raw = y_test_raw.copy()

print("Kích thước train sau xử lý ngoại lai:")
display_shape("X_train_final", X_train_final)
print("y_train_final_raw:", y_train_final_raw.shape)

OUTLIER_MODE: keep


,set,lower_bound_fit_on_train,upper_bound_fit_on_train,n,n_outliers,outlier_rate
0,train,"44,013.0000","244,871.0000",200000,1849,0.0092
1,test,"44,013.0000","244,871.0000",50000,467,0.0093


Kích thước train sau xử lý ngoại lai:
X_train_final: shape = (200000, 9)
y_train_final_raw: (200000,)


In [10]:
# ============================================================
# CELL 10. BIẾN ĐỔI TARGET
# ============================================================

y_train_final = transform_target(y_train_final_raw, mode=TARGET_MODE)
y_test_final = transform_target(y_test_final_raw, mode=TARGET_MODE)

print("TARGET_MODE:", TARGET_MODE)
print("y_train_final mô tả:")
display(y_train_final.describe())
print("y_test_final mô tả:")
display(y_test_final.describe())

TARGET_MODE: raw
y_train_final mô tả:


count   200,000.0000
mean    145,713.8819
std      37,409.7244
min      31,867.0000
25%     119,334.7500
50%     143,430.0000
75%     169,549.2500
max     333,046.0000
Name: salary, dtype: float64

y_test_final mô tả:


count    50,000.0000
mean    145,734.8752
std      37,401.2345
min      37,213.0000
25%     119,448.7500
50%     143,539.5000
75%     169,258.2500
max     327,217.0000
Name: salary, dtype: float64

In [11]:
# ============================================================
# CELL 11. XÁC ĐỊNH BIẾN SỐ / BIẾN PHÂN LOẠI SAU KHI CHIA TRAIN
# ============================================================

# Xác định lại trên X_train_final để pipeline chỉ fit theo train.
numeric_features = X_train_final.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = [c for c in X_train_final.columns if c not in numeric_features]

print("Numeric features:")
print(numeric_features)
print("\nCategorical features:")
print(categorical_features)

feature_type_report = pd.DataFrame({
    "feature": numeric_features + categorical_features,
    "type_for_preprocessing": ["numeric"] * len(numeric_features) + ["categorical"] * len(categorical_features)
})

feature_type_report.to_csv(OUTPUT_DIR / "03_feature_type_report.csv", index=False, encoding="utf-8-sig")
display(feature_type_report)

Numeric features:
['experience_years', 'skills_count', 'certifications']

Categorical features:
['job_title', 'education_level', 'industry', 'company_size', 'location', 'remote_work']


,feature,type_for_preprocessing
0,experience_years,numeric
1,skills_count,numeric
2,certifications,numeric
3,job_title,categorical
4,education_level,categorical
5,industry,categorical
6,company_size,categorical
7,location,categorical
8,remote_work,categorical


In [12]:
# ============================================================
# CELL 12. XÂY DỰNG PIPELINE TIỀN XỬ LÝ
# ============================================================

if SCALE_NUMERIC:
    numeric_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
else:
    numeric_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", make_onehot_encoder())
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ],
    remainder="drop",
    sparse_threshold=0.3
)

print(preprocessor)

ColumnTransformer(transformers=[('numeric',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['experience_years', 'skills_count',
                                  'certifications']),
                                ('categorical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['job_title', 'education_level', 'industry',
                                  'company_size', 'location', 'remote_work'])])


In [13]:
# ============================================================
# CELL 13. FIT PIPELINE TRÊN TRAIN VÀ TRANSFORM TRAIN/TEST
# ============================================================

X_train_processed = preprocessor.fit_transform(X_train_final)
X_test_processed = preprocessor.transform(X_test_final)

feature_names = get_feature_names(preprocessor, numeric_features, categorical_features)

print("Dữ liệu sau tiền xử lý:")
display_shape("X_train_processed", X_train_processed)
display_shape("X_test_processed", X_test_processed)
print("Số tên đặc trưng sau xử lý:", len(feature_names))
print("Kiểu dữ liệu X_train_processed:", type(X_train_processed))

feature_names_df = pd.DataFrame({
    "feature_index": range(len(feature_names)),
    "feature_name": feature_names
})

feature_names_df.to_csv(OUTPUT_DIR / "04_processed_feature_names.csv", index=False, encoding="utf-8-sig")
display(feature_names_df.head(30))

Dữ liệu sau tiền xử lý:
X_train_processed: shape = (200000, 48)
X_test_processed: shape = (50000, 48)
Số tên đặc trưng sau xử lý: 48
Kiểu dữ liệu X_train_processed: <class 'scipy.sparse._csr.csr_matrix'>


,feature_index,feature_name
0,0,experience_years
1,1,skills_count
2,2,certifications
3,3,job_title_AI Engineer
4,4,job_title_Backend Developer
5,5,job_title_Business Analyst
6,6,job_title_Cloud Engineer
7,7,job_title_Cybersecurity Analyst
8,8,job_title_Data Analyst
9,9,job_title_Data Scientist


In [14]:
# ============================================================
# CELL 14. KIỂM TRA SAU TIỀN XỬ LÝ
# ============================================================

def count_missing_in_processed_matrix(X_processed):
    """
    Đếm NaN trong ma trận dense/sparse.
    """
    if sparse.issparse(X_processed):
        data = X_processed.data
        return int(np.isnan(data).sum())
    return int(np.isnan(X_processed).sum())

missing_train_processed = count_missing_in_processed_matrix(X_train_processed)
missing_test_processed = count_missing_in_processed_matrix(X_test_processed)

post_report = pd.DataFrame({
    "item": [
        "n_train_rows",
        "n_test_rows",
        "n_original_features",
        "n_processed_features",
        "missing_in_X_train_processed",
        "missing_in_X_test_processed",
        "target_mode",
        "outlier_mode",
        "scale_numeric"
    ],
    "value": [
        X_train_processed.shape[0],
        X_test_processed.shape[0],
        len(feature_cols),
        len(feature_names),
        missing_train_processed,
        missing_test_processed,
        TARGET_MODE,
        OUTLIER_MODE,
        SCALE_NUMERIC
    ]
})

post_report.to_csv(OUTPUT_DIR / "05_post_preprocessing_report.csv", index=False, encoding="utf-8-sig")

display(post_report)

,item,value
0,n_train_rows,200000
1,n_test_rows,50000
2,n_original_features,9
3,n_processed_features,48
4,missing_in_X_train_processed,0
5,missing_in_X_test_processed,0
6,target_mode,raw
7,outlier_mode,keep
8,scale_numeric,True


In [15]:
# ============================================================
# CELL 15. LƯU DỮ LIỆU ĐÃ TIỀN XỬ LÝ VÀ PIPELINE
# ============================================================

# 1. Lưu pipeline tiền xử lý.
preprocessor_path = OUTPUT_DIR / "preprocessor.joblib"
joblib.dump(preprocessor, preprocessor_path)

# 2. Lưu X_train, X_test dạng sparse .npz nếu có thể.
# Nếu X đang dense, chuyển về sparse để lưu gọn hơn.
X_train_sparse = X_train_processed if sparse.issparse(X_train_processed) else sparse.csr_matrix(X_train_processed)
X_test_sparse = X_test_processed if sparse.issparse(X_test_processed) else sparse.csr_matrix(X_test_processed)

sparse.save_npz(OUTPUT_DIR / "X_train_processed.npz", X_train_sparse)
sparse.save_npz(OUTPUT_DIR / "X_test_processed.npz", X_test_sparse)

# 3. Lưu target.
pd.DataFrame({
    "y_train_raw_salary": y_train_final_raw.reset_index(drop=True),
    "y_train_model_target": pd.Series(y_train_final).reset_index(drop=True)
}).to_csv(OUTPUT_DIR / "y_train.csv", index=False, encoding="utf-8-sig")

pd.DataFrame({
    "y_test_raw_salary": y_test_final_raw.reset_index(drop=True),
    "y_test_model_target": pd.Series(y_test_final).reset_index(drop=True)
}).to_csv(OUTPUT_DIR / "y_test.csv", index=False, encoding="utf-8-sig")

# 4. Lưu dữ liệu thô đã chia train/test để truy vết.
X_train_final.reset_index(drop=True).to_csv(OUTPUT_DIR / "X_train_raw_split.csv", index=False, encoding="utf-8-sig")
X_test_final.reset_index(drop=True).to_csv(OUTPUT_DIR / "X_test_raw_split.csv", index=False, encoding="utf-8-sig")

# 5. Lưu metadata cấu hình.
metadata = {
    "data_path": str(DATA_PATH),
    "target": TARGET,
    "target_mode": TARGET_MODE,
    "test_size": TEST_SIZE,
    "random_state": RANDOM_STATE,
    "outlier_mode": OUTLIER_MODE,
    "iqr_k": IQR_K,
    "create_experience_group": CREATE_EXPERIENCE_GROUP,
    "use_experience_group_as_feature": USE_EXPERIENCE_GROUP_AS_FEATURE,
    "scale_numeric": SCALE_NUMERIC,
    "exclude_features": exclude_features,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "n_train": int(X_train_final.shape[0]),
    "n_test": int(X_test_final.shape[0]),
    "n_processed_features": int(len(feature_names)),
    "processed_feature_names_file": "04_processed_feature_names.csv",
    "preprocessor_file": "preprocessor.joblib"
}

save_json(metadata, OUTPUT_DIR / "preprocessing_metadata.json")

print("Đã lưu pipeline:", preprocessor_path)
print("Đã lưu toàn bộ kết quả vào thư mục:", OUTPUT_DIR.resolve())

Đã lưu pipeline: preprocessing_salary_outputs\preprocessor.joblib
Đã lưu toàn bộ kết quả vào thư mục: C:\Users\AD\OneDrive\Tài liệu\2025.2\DSS\preprocessing_salary_outputs


In [16]:
# ============================================================
# CELL 16. TÙY CHỌN: LƯU CSV DENSE SAU TIỀN XỬ LÝ
# ============================================================

# Chỉ bật SAVE_DENSE_PROCESSED_CSV = True nếu cần xem dữ liệu dạng bảng.
# Với dữ liệu lớn, nên dùng .npz + feature_names để huấn luyện mô hình.

if SAVE_DENSE_PROCESSED_CSV:
    if sparse.issparse(X_train_processed):
        X_train_dense = X_train_processed.toarray()
        X_test_dense = X_test_processed.toarray()
    else:
        X_train_dense = np.asarray(X_train_processed)
        X_test_dense = np.asarray(X_test_processed)

    train_processed_df = pd.DataFrame(X_train_dense, columns=feature_names)
    test_processed_df = pd.DataFrame(X_test_dense, columns=feature_names)

    train_processed_df[TARGET if TARGET_MODE == "raw" else "log_salary"] = pd.Series(y_train_final).reset_index(drop=True)
    test_processed_df[TARGET if TARGET_MODE == "raw" else "log_salary"] = pd.Series(y_test_final).reset_index(drop=True)

    train_processed_df.to_csv(OUTPUT_DIR / "train_processed_dense.csv", index=False, encoding="utf-8-sig")
    test_processed_df.to_csv(OUTPUT_DIR / "test_processed_dense.csv", index=False, encoding="utf-8-sig")

    print("Đã lưu train_processed_dense.csv và test_processed_dense.csv")
else:
    print("Không lưu CSV dense. Đang dùng định dạng sparse .npz để tiết kiệm dung lượng.")

Không lưu CSV dense. Đang dùng định dạng sparse .npz để tiết kiệm dung lượng.


In [17]:
# ============================================================
# CELL 17. CÁCH NẠP LẠI DỮ LIỆU ĐÃ XỬ LÝ Ở NOTEBOOK MÔ HÌNH
# ============================================================

# Cell này minh họa cách dùng kết quả tiền xử lý ở notebook huấn luyện mô hình.
# Khi sang bước mô hình hóa, bạn có thể copy đoạn này.

loaded_preprocessor = joblib.load(OUTPUT_DIR / "preprocessor.joblib")
loaded_X_train = sparse.load_npz(OUTPUT_DIR / "X_train_processed.npz")
loaded_X_test = sparse.load_npz(OUTPUT_DIR / "X_test_processed.npz")
loaded_y_train = pd.read_csv(OUTPUT_DIR / "y_train.csv")["y_train_model_target"]
loaded_y_test = pd.read_csv(OUTPUT_DIR / "y_test.csv")["y_test_model_target"]
loaded_feature_names = pd.read_csv(OUTPUT_DIR / "04_processed_feature_names.csv")

print("loaded_X_train:", loaded_X_train.shape)
print("loaded_X_test:", loaded_X_test.shape)
print("loaded_y_train:", loaded_y_train.shape)
print("loaded_y_test:", loaded_y_test.shape)

display(loaded_feature_names.head())

loaded_X_train: (200000, 48)
loaded_X_test: (50000, 48)
loaded_y_train: (200000,)
loaded_y_test: (50000,)


,feature_index,feature_name
0,0,experience_years
1,1,skills_count
2,2,certifications
3,3,job_title_AI Engineer
4,4,job_title_Backend Developer


In [18]:
# ============================================================
# CELL 18. HÀM TIỀN XỬ LÝ HỒ SƠ MỚI CHO DSS
# ============================================================

# Sau khi hệ thống có người dùng nhập hồ sơ mới, không được fit lại encoder/scaler.
# Chỉ dùng loaded_preprocessor.transform(new_profile).


def preprocess_new_profiles(new_profiles, fitted_preprocessor, expected_raw_columns):
    """
    Tiền xử lý hồ sơ người dùng mới bằng pipeline đã fit trên train.

    Parameters
    ----------
    new_profiles : dict hoặc pd.DataFrame
        Hồ sơ mới. Có thể truyền một dict hoặc bảng nhiều hồ sơ.
    fitted_preprocessor : ColumnTransformer
        Pipeline đã fit.
    expected_raw_columns : list
        Danh sách cột đầu vào ban đầu của mô hình.

    Returns
    -------
    X_new_processed : sparse matrix hoặc ndarray
        Hồ sơ mới sau tiền xử lý, sẵn sàng đưa vào mô hình dự đoán.
    """
    if isinstance(new_profiles, dict):
        new_profiles = pd.DataFrame([new_profiles])
    else:
        new_profiles = new_profiles.copy()

    new_profiles = standardize_columns(new_profiles)
    new_profiles = clean_categorical_strings(new_profiles)

    if CREATE_EXPERIENCE_GROUP:
        new_profiles = add_experience_group(new_profiles, experience_col="experience_years")

    # Bổ sung cột thiếu bằng NaN để pipeline tự impute.
    for col in expected_raw_columns:
        if col not in new_profiles.columns:
            new_profiles[col] = np.nan

    # Giữ đúng thứ tự cột như khi train.
    new_profiles = new_profiles[expected_raw_columns]

    return fitted_preprocessor.transform(new_profiles)


# Ví dụ hồ sơ mới.
example_profile = {
    "job_title": "Data Analyst",
    "experience_years": 5,
    "education_level": "Bachelor",
    "skills_count": 12,
    "industry": "Finance",
    "company_size": "Medium",
    "location": "Singapore",
    "remote_work": "Hybrid",
    "certifications": 2
}

X_new = preprocess_new_profiles(
    new_profiles=example_profile,
    fitted_preprocessor=loaded_preprocessor,
    expected_raw_columns=feature_cols
)

print("Hồ sơ mới sau tiền xử lý:", X_new.shape)

Hồ sơ mới sau tiền xử lý: (1, 48)


## Kết luận tiền xử lý

Sau notebook này, dữ liệu đã sẵn sàng để huấn luyện mô hình dự đoán lương. Các kết quả chính được lưu trong thư mục `preprocessing_salary_outputs`:

- `preprocessor.joblib`: pipeline tiền xử lý đã fit trên train set.
- `X_train_processed.npz`, `X_test_processed.npz`: ma trận đặc trưng sau chuẩn hóa và One-Hot Encoding.
- `y_train.csv`, `y_test.csv`: biến mục tiêu tương ứng.
- `04_processed_feature_names.csv`: tên các đặc trưng sau tiền xử lý.
- `preprocessing_metadata.json`: cấu hình tiền xử lý.
- `01_preprocessing_quality_report.csv`, `05_post_preprocessing_report.csv`: báo cáo kiểm tra trước và sau tiền xử lý.

Điểm quan trọng nhất là pipeline được fit trên tập train trước, sau đó mới transform tập test. Cách làm này tránh rò rỉ dữ liệu và phù hợp với quy trình mô hình hóa trong hệ hỗ trợ quyết định.